In [4]:
import json
import re

In [5]:
def fix_latex(input_file, output_file):
    """
    Fix Unicode escape sequences and convert special characters to LaTeX in JSONL file.
    
    Fixes:
    - Unicode escape sequences like \\u00d7 to proper characters or LaTeX
    - Greek letters to LaTeX commands (α -> \\alpha, β -> \\beta, etc.)
    - Math symbols to LaTeX (× -> \\times, ≈ -> \\approx, etc.)
    - HTML tags like <sup> to LaTeX ^{}
    - Exponential notation patterns (e-λt -> e^{-\\lambda t})
    - Unicode minus sign to regular minus
    """
    
    # Mapping for Unicode to LaTeX conversion
    unicode_to_latex = {
        # Greek letters
        'α': r'\alpha',
        'β': r'\beta',
        'λ': r'\lambda',
        'η': r'\eta',
        'Γ': r'\Gamma',
        'θ': r'\theta',
        'μ': r'\mu',
        'σ': r'\sigma',
        'χ': r'\chi',
        
        # Math symbols
        '×': r'\times',
        '≈': r'\approx',
        '≤': r'\leq',
        '≥': r'\geq',
        '∞': r'\infty',
        '∫': r'\int',
        '±': r'\pm',
        
        # Unicode minus to regular minus
        '−': '-',
        
        # Em dash to regular dash
        '—': '--',
        
        # Degree symbol
        '°': r'^\circ',
        
        # Superscript numbers
        '²': '^2',
        
        # Subscript numbers
        'ₕ': '_h',
        '₀': '_0',
        '₁': '_1',
        '₂': '_2',
        '₃': '_3',
        '₄': '_4',
        '₅': '_5',
        '₆': '_6',
        '₇': '_7',
        '₈': '_8',
        '₉': '_9',
    }
    
    def convert_to_latex(text):
        """Apply all LaTeX conversions to text."""
        
        # 1. Convert HTML tags
        text = re.sub(r'<sup>(.*?)</sup>', r'^{\1}', text)
        text = re.sub(r'<sub>(.*?)</sub>', r'_{\1}', text)
        
        # 2. Convert Unicode characters
        for unicode_char, latex_cmd in unicode_to_latex.items():
            text = text.replace(unicode_char, latex_cmd)
        
        # 3. Clean up markdown asterisks around LaTeX
        text = re.sub(r'\*\^{([^}]+)}\*', r'^{\1}', text)
        text = re.sub(r'\*_{([^}]+)}\*', r'_{\1}', text)
        
        # 4. Fix exponential patterns like e-\\lambda^{t} -> e^{-\\lambda t}
        greek_letters = ['lambda', 'beta', 'alpha', 'theta', 'eta', 'mu', 'sigma']
        for letter in greek_letters:
            text = re.sub(rf'e-\\{letter}\*?\^{{([^}}]+)}}\*?', rf'e^{{-\\{letter} \1}}', text)
            text = re.sub(rf'e-\\{letter}\^{{([^}}]+)}}', rf'e^{{-\\{letter} \1}}', text)
        
        # 5. Add spaces after Greek letters when followed by lowercase letter
        for letter in greek_letters + ['gamma', 'chi']:
            text = re.sub(rf'\\{letter}([a-z_])', rf'\\{letter} \1', text)
        
        return text
    
    def process_obj(obj):
        """Recursively process JSON object."""
        if isinstance(obj, dict):
            return {k: process_obj(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [process_obj(item) for item in obj]
        elif isinstance(obj, str):
            return convert_to_latex(obj)
        return obj
    
    # Process file
    with open(input_file, 'r', encoding='utf-8') as f_in:
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for line_num, line in enumerate(f_in, 1):
                try:
                    obj = json.loads(line)
                    obj = process_obj(obj)
                    json.dump(obj, f_out, ensure_ascii=False)
                    f_out.write('\n')
                except Exception as e:
                    print(f"Error on line {line_num}: {e}")
                    f_out.write(line)
    
    print(f"Processed {line_num} lines -> {output_file}")

In [6]:
fix_latex('../content/dataset_reliability_verified.jsonl', '../content/dataset_reliability_verified_cleaned.jsonl')

Processed 68 lines -> ../content/dataset_reliability_verified_cleaned.jsonl
